Stagnation metrics

In [ ]:
%pip install pandas requests beautifulsoup4 edgartools numpy torch scikit-learn transformers sentence-transformers

1) Fetching tickers

import requests
import pandas as pd
from bs4 import BeautifulSoup

OUTPUT_FILE = "sp500_tickers.csv"

def get_sp500_tickers_bs4():
    url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/91.0.4472.124 Safari/537.36"
        )
    }

    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")
        table = soup.find("table", {"id": "constituents"})

        if not table:
            print("Could not find S&P 500 table.")
            return []

        tickers = []
        rows = table.find("tbody").find_all("tr")

        for row in rows[1:]:
            cells = row.find_all("td")
            if len(cells) < 1:
                continue

            ticker = cells[0].get_text(strip=True).replace(".", "-")
            tickers.append(ticker)

        print(f"Retrieved {len(tickers)} S&P 500 tickers.")
        return tickers

    except requests.exceptions.RequestException as e:
        print(f"Request failed: {e}")
        return []


def main():
    print("=== STEP 1: FETCHING S&P 500 TICKERS ===\n")

    tickers = get_sp500_tickers_bs4()

    if not tickers:
        print("No tickers retrieved. Exiting.")
        return

    for i, ticker in enumerate(tickers, 1):
        print(f"{i}. {ticker}")

    pd.DataFrame({"ticker": tickers}).to_csv(OUTPUT_FILE, index=False)

    return tickers

tickers = main()

2) MD&A text extraction

In [ ]:
import re
import time
import queue
import requests
import multiprocessing as mp

from urllib.parse import urljoin, urlparse, parse_qs
import pandas as pd
from bs4 import BeautifulSoup
from edgar import Company, set_identity


set_identity("oscarchanly@gmail.com")

sec_headers = {
    "User-Agent": "Dummy Company oscarchanly@gmail.com",
    "Accept-Encoding": "gzip, deflate",
    "Host": "www.sec.gov",
}


# 1. Get full 10-K filings only

def get_n_year_full_10k_filings(ticker, n_years, retrieval_multiplier):
    """
    Fetch latest full 10-K filings only.
    Excludes 10-K/A amendments.
    """
    company = Company(ticker)

    raw_filings = company.get_filings(form="10-K").latest(
        n_years * retrieval_multiplier
    )

    raw_filings = list(raw_filings)
    full_10k_filings = []

    for filing in raw_filings:
        form = str(getattr(filing, "form", "")).strip()

        if form == "10-K":
            full_10k_filings.append(filing)

        if len(full_10k_filings) >= n_years:
            break

    return full_10k_filings


# 2. Text normalization

def normalize_text(text):
    """
    Clean SEC filing text for regex matching.
    """
    if not isinstance(text, str):
        return ""

    text = text.replace("\u00A0", " ")
    text = text.replace("’", "'").replace("‘", "'").replace("`", "'")
    text = text.replace("“", '"').replace("”", '"')
    text = text.replace("–", "-").replace("—", "-")
    text = text.replace("&nbsp;", " ")

    text = re.sub(r"\s+", " ", text).strip()

    return text


def html_to_clean_text(html):
    """
    Convert HTML to clean readable text.
    """
    if not html:
        return ""

    soup = BeautifulSoup(html, "html.parser")

    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()

    text = soup.get_text(separator=" ")
    text = normalize_text(text)

    return text


# 3. SEC URL helpers

def clean_sec_url(url):
    """
    Convert SEC ixviewer URLs into direct document URLs where possible.
    """
    if not isinstance(url, str) or not url.strip():
        return None

    url = url.strip()

    if "ixviewer/doc/action" in url and "doc=" in url:
        parsed = urlparse(url)
        query = parse_qs(parsed.query)
        doc_path = query.get("doc", [None])[0]

        if doc_path:
            return urljoin("https://www.sec.gov", doc_path)

    return url


def get_candidate_urls_from_filing(filing):
    """
    Collect possible URLs from the edgar Filing object.
    """
    candidate_urls = []

    url_attrs = [
        "filing_url",
        "document_url",
        "primary_document_url",
        "html_url",
    ]

    for attr in url_attrs:
        value = getattr(filing, attr, None)

        if isinstance(value, str) and value.strip():
            candidate_urls.append(value.strip())

    filing_url = getattr(filing, "filing_url", None)
    primary_document = getattr(filing, "primary_document", None)

    if isinstance(primary_document, str) and filing_url:
        base_url = filing_url.rsplit("/", 1)[0] + "/"
        candidate_urls.append(urljoin(base_url, primary_document))

    if primary_document is not None and not isinstance(primary_document, str):
        for attr in ["url", "document_url", "href"]:
            value = getattr(primary_document, attr, None)

            if isinstance(value, str) and value.strip():
                candidate_urls.append(value.strip())

    cleaned_urls = []

    for url in candidate_urls:
        cleaned = clean_sec_url(url)

        if cleaned and cleaned not in cleaned_urls:
            cleaned_urls.append(cleaned)

    return cleaned_urls


def download_html(url, request_timeout_seconds):
    """
    Download HTML/text from SEC.
    """
    if not url:
        return None

    try:
        response = requests.get(
            url,
            headers=sec_headers,
            timeout=request_timeout_seconds,
        )

        response.raise_for_status()

        return response.text

    except Exception:
        return None


def find_primary_10k_document_url(index_html, index_url):
    """
    If filing_url points to a SEC filing detail page, find the actual primary 10-K document.
    """
    if not index_html:
        return None

    lower_html = index_html.lower()

    if "document format files" not in lower_html and "sec filing" not in lower_html:
        return None

    soup = BeautifulSoup(index_html, "html.parser")

    for row in soup.find_all("tr"):
        cells = [cell.get_text(" ", strip=True) for cell in row.find_all(["td", "th"])]
        row_text = " ".join(cells).upper()

        if "10-K/A" in row_text:
            continue

        if re.search(r"\b10-K\b", row_text):
            links = row.find_all("a", href=True)

            for link in links:
                href = link.get("href", "")

                if not href:
                    continue

                href_lower = href.lower()

                if "ixviewer" in href_lower:
                    href = clean_sec_url(urljoin(index_url, href))

                if href_lower.endswith((".htm", ".html", ".txt")):
                    return urljoin(index_url, href)

    return None


# 4. Get best filing text

def filing_text_quality_score(text):
    """
    Choose the best text source for a filing.
    Higher score means more likely to be the actual 10-K body.
    """
    if not text:
        return 0

    lower = text.lower()

    item7_count = len(re.findall(r"\bitem\s*7(?!\s*a)", lower))
    item7a_count = len(re.findall(r"\bitem\s*7a", lower))
    item8_count = len(re.findall(r"\bitem\s*8", lower))

    mda_phrase_count = len(
        re.findall(
            r"management'?s\s+discussion\s+(?:and|&)\s+analysis",
            lower,
        )
    )

    score = len(text)
    score += item7_count * 100_000
    score += item7a_count * 50_000
    score += item8_count * 50_000
    score += mda_phrase_count * 200_000

    return score


def get_filing_text(filing, request_timeout_seconds):
    """
    Download and select the best available text version of the filing.

    This tries:
    1. direct filing/document URLs
    2. primary document link from SEC index page
    3. edgar filing.html() / filing.text() fallbacks
    """
    text_candidates = []

    candidate_urls = get_candidate_urls_from_filing(filing)

    for url in candidate_urls:
        html = download_html(url, request_timeout_seconds)

        if not html:
            continue

        primary_doc_url = find_primary_10k_document_url(html, url)

        if primary_doc_url and primary_doc_url != url:
            primary_html = download_html(primary_doc_url, request_timeout_seconds)

            if primary_html:
                primary_text = html_to_clean_text(primary_html)

                if primary_text:
                    text_candidates.append(primary_text)

        direct_text = html_to_clean_text(html)

        if direct_text:
            text_candidates.append(direct_text)

    for method_name in ["html", "text"]:
        method = getattr(filing, method_name, None)

        if callable(method):
            try:
                content = method()

                if content:
                    content = str(content)
                    text = html_to_clean_text(content)

                    if text:
                        text_candidates.append(text)

            except Exception:
                pass

    if not text_candidates:
        return None

    best_text = max(text_candidates, key=filing_text_quality_score)

    return best_text if best_text else None


# 5. MD&A candidate scoring

def score_mda_candidate(candidate_text, start_idx, full_text_length, toc_max_chars):
    """
    Score an Item 7 candidate to avoid table-of-contents matches.
    """
    if not candidate_text:
        return 0, True, 0

    lower = candidate_text.lower()
    candidate_len = len(candidate_text)

    content_keywords = [
        "overview",
        "results of operations",
        "financial condition",
        "liquidity",
        "capital resources",
        "cash flows",
        "net sales",
        "revenue",
        "sales",
        "gross profit",
        "operating income",
        "operating results",
        "year ended",
        "fiscal",
        "we expect",
        "we believe",
        "compared to",
        "increase",
        "decrease",
        "critical accounting",
        "cash and cash equivalents",
        "management's discussion",
        "discussion and analysis",
    ]

    content_score = sum(
        1 for keyword in content_keywords
        if keyword in lower
    )

    early_in_document = (
        full_text_length > 0
        and start_idx < full_text_length * 0.15
    )

    toc_like_terms = [
        "table of contents",
        "part i",
        "part ii",
        "page",
    ]

    toc_term_score = sum(
        1 for term in toc_like_terms
        if term in lower[:1000]
    )

    is_likely_toc = (
        early_in_document
        and candidate_len < toc_max_chars
        and content_score < 2
    )

    if candidate_len < toc_max_chars and toc_term_score >= 2 and content_score < 3:
        is_likely_toc = True

    candidate_score = candidate_len + (content_score * 1500)

    if is_likely_toc:
        candidate_score -= 10000

    return content_score, is_likely_toc, candidate_score


# 6. Extract MD&A from text

def find_mda_start_matches(lower_text):
    """
    Find possible true Item 7 MD&A starts.
    """
    item_sep = r"\s*[\.\:\-]?\s*"

    start_patterns = [
        rf"\bitem\s*7(?!\s*a){item_sep}management'?s\s+discussion\s+(?:and|&)\s+analysis",
        rf"\bitem\s*7(?!\s*a){item_sep}managements\s+discussion\s+(?:and|&)\s+analysis",
        rf"\bitem\s*7(?!\s*a){item_sep}management\s+s\s+discussion\s+(?:and|&)\s+analysis",
        rf"\bitem\s*7(?!\s*a){item_sep}management\s+discussion\s+(?:and|&)\s+analysis",
        rf"\bitem\s*7(?!\s*a){item_sep}discussion\s+(?:and|&)\s+analysis",
        rf"\bitem\s*7(?!\s*a){item_sep}management'?s\s+discussion",
        rf"\bitem\s*7(?!\s*a){item_sep}managements\s+discussion",
        rf"\bitem\s*7(?!\s*a){item_sep}md\s*&\s*a",
        rf"\bitem\s*7(?!\s*a){item_sep}m\s*d\s*&\s*a",
    ]

    start_matches = []

    for pattern in start_patterns:
        start_matches.extend(list(re.finditer(pattern, lower_text)))

    unique_starts = {}

    for match in start_matches:
        unique_starts[match.start()] = match

    start_matches = sorted(
        unique_starts.values(),
        key=lambda match: match.start()
    )

    return start_matches


def find_mda_end_indices(lower_text, start_idx):
    """
    Find possible Item 7A or Item 8 ending boundaries.
    Uses specific heading patterns first, then cautious fallback patterns.
    """
    item_sep = r"\s*[\.\:\-]?\s*"

    search_area = lower_text[start_idx:]

    specific_end_patterns = [
        rf"\bitem\s*7a{item_sep}quantitative",
        rf"\bitem\s*7a{item_sep}qualitative",
        rf"\bitem\s*7a{item_sep}quantitative\s+(?:and|&)\s+qualitative",
        rf"\bitem\s*7a{item_sep}market\s+risk",
        rf"\bitem\s*7a{item_sep}controls",
        rf"\bitem\s*8{item_sep}financial\s+statements",
        rf"\bitem\s*8{item_sep}financial\s+statement",
        rf"\bitem\s*8{item_sep}consolidated\s+financial",
        rf"\bitem\s*8{item_sep}audited\s+financial",
        rf"\bitem\s*8{item_sep}supplementary\s+data",
    ]

    possible_end_indices = []

    for pattern in specific_end_patterns:
        for match in re.finditer(pattern, search_area):
            possible_end_idx = start_idx + match.start()

            if possible_end_idx > start_idx:
                possible_end_indices.append(possible_end_idx)

    if possible_end_indices:
        return possible_end_indices

    cautious_fallback_patterns = [
        r"\bitem\s*7a\b",
        r"\bitem\s*8\b",
    ]

    for pattern in cautious_fallback_patterns:
        for match in re.finditer(pattern, search_area):
            possible_end_idx = start_idx + match.start()

            if possible_end_idx <= start_idx:
                continue

            gap = possible_end_idx - start_idx

            if gap < 500:
                continue

            before = lower_text[max(0, possible_end_idx - 60):possible_end_idx]
            after = lower_text[possible_end_idx:possible_end_idx + 250]

            reference_phrases = [
                "see ",
                "refer to ",
                "included in ",
                "described in ",
                "discussed in ",
                "under ",
            ]

            if any(phrase in before for phrase in reference_phrases):
                continue

            if "item 7a" in after:
                if any(term in after for term in ["quantitative", "qualitative", "market risk"]):
                    possible_end_indices.append(possible_end_idx)

            elif "item 8" in after:
                if any(term in after for term in ["financial", "consolidated", "statements", "supplementary"]):
                    possible_end_indices.append(possible_end_idx)

    return possible_end_indices


def extract_mda_from_text(
    text,
    min_chars,
    absolute_min_mda_chars,
    toc_max_chars,
):
    """
    Extract Item 7 MD&A section only.
    Explicitly cuts before Item 7A or Item 8.
    """
    if not text:
        return None

    text = normalize_text(text)
    lower_text = text.lower()
    full_text_length = len(text)

    start_matches = find_mda_start_matches(lower_text)

    if not start_matches:
        return None

    candidates = []

    item_sep = r"\s*[\.\:\-]?\s*"

    for start_match in start_matches:
        start_idx = start_match.start()

        possible_end_indices = find_mda_end_indices(
            lower_text=lower_text,
            start_idx=start_idx,
        )

        if not possible_end_indices:
            continue

        end_idx = min(possible_end_indices)

        if end_idx <= start_idx:
            continue

        mda_candidate = text[start_idx:end_idx].strip()

        mda_candidate = re.sub(
            rf"^\s*item\s*7(?!\s*a){item_sep}",
            "",
            mda_candidate,
            flags=re.IGNORECASE,
        ).strip()

        candidate_len = len(mda_candidate)

        content_score, is_likely_toc, candidate_score = score_mda_candidate(
            candidate_text=mda_candidate,
            start_idx=start_idx,
            full_text_length=full_text_length,
            toc_max_chars=toc_max_chars,
        )

        if is_likely_toc:
            continue

        valid_by_length = candidate_len >= min_chars

        valid_short_but_content_rich = (
            candidate_len >= absolute_min_mda_chars
            and content_score >= 2
        )

        if valid_by_length or valid_short_but_content_rich:
            candidates.append({
                "text": mda_candidate,
                "candidate_score": candidate_score,
            })

    if not candidates:
        return None

    best_candidate = max(
        candidates,
        key=lambda candidate: candidate["candidate_score"]
    )

    return best_candidate["text"]


# 7. Process one ticker

def process_one_ticker_mda(
    ticker,
    n_years,
    min_chars,
    request_timeout_seconds,
    absolute_min_mda_chars,
    toc_max_chars,
    retrieval_multiplier,
):
    """
    Extract MD&A text from recent full 10-K filings for one ticker.
    Returns successful rows only.
    """
    rows = []

    try:
        filings = get_n_year_full_10k_filings(
            ticker=ticker,
            n_years=n_years,
            retrieval_multiplier=retrieval_multiplier,
        )

        for filing in filings:
            filing_date = getattr(filing, "filing_date", None)

            filing_year = (
                filing_date.year
                if hasattr(filing_date, "year")
                else None
            )

            filing_url = getattr(filing, "filing_url", None)

            filing_text = get_filing_text(
                filing=filing,
                request_timeout_seconds=request_timeout_seconds,
            )

            mda_text = extract_mda_from_text(
                text=filing_text,
                min_chars=min_chars,
                absolute_min_mda_chars=absolute_min_mda_chars,
                toc_max_chars=toc_max_chars,
            )

            if mda_text is not None:
                rows.append({
                    "ticker": ticker,
                    "year": filing_year,
                    "filing_url": filing_url,
                    "mda_char_length": len(mda_text),
                    "mda_text": mda_text,
                })

    except Exception:
        pass

    return rows


# 8. Multiprocessing worker

def ticker_worker(
    job_id,
    ticker,
    n_years,
    min_chars,
    request_timeout_seconds,
    absolute_min_mda_chars,
    toc_max_chars,
    retrieval_multiplier,
    result_queue,
):
    """
    Run one ticker extraction inside a multiprocessing worker.
    Returns successful extraction rows only.
    """
    try:
        rows = process_one_ticker_mda(
            ticker=ticker,
            n_years=n_years,
            min_chars=min_chars,
            request_timeout_seconds=request_timeout_seconds,
            absolute_min_mda_chars=absolute_min_mda_chars,
            toc_max_chars=toc_max_chars,
            retrieval_multiplier=retrieval_multiplier,
        )

        result_queue.put({
            "job_id": job_id,
            "ticker": ticker,
            "rows": rows,
        })

    except Exception:
        result_queue.put({
            "job_id": job_id,
            "ticker": ticker,
            "rows": [],
        })


# 9. Run extraction 

def run_mda_extraction_clean(
    tickers,
    n_years,
    min_chars,
    max_workers,
    ticker_timeout_seconds,
    request_timeout_seconds,
    absolute_min_mda_chars,
    toc_max_chars,
    retrieval_multiplier,
    output_filename,
):
    """
    Run MD&A extraction across all tickers.

    Output:
    One clean CSV only:
    ticker, year, filing_url, mda_char_length, mda_text
    """
    tickers = list(tickers)
    all_rows = []

    try:
        ctx = mp.get_context("fork")
    except ValueError:
        ctx = mp.get_context()

    result_queue = ctx.Queue()

    active_jobs = {}
    timed_out_jobs = set()

    next_job_id = 0
    ticker_index = 0
    total_tickers = len(tickers)

    print(f"Starting MD&A extraction for {total_tickers} tickers...")

    while ticker_index < total_tickers or active_jobs:

        while ticker_index < total_tickers and len(active_jobs) < max_workers:
            ticker = tickers[ticker_index]
            ticker_index += 1
            next_job_id += 1

            process = ctx.Process(
                target=ticker_worker,
                args=(
                    next_job_id,
                    ticker,
                    n_years,
                    min_chars,
                    request_timeout_seconds,
                    absolute_min_mda_chars,
                    toc_max_chars,
                    retrieval_multiplier,
                    result_queue,
                ),
            )

            process.start()

            active_jobs[next_job_id] = {
                "ticker": ticker,
                "process": process,
                "start_time": time.time(),
            }

        while True:
            try:
                result = result_queue.get_nowait()
            except queue.Empty:
                break

            job_id = result.get("job_id")
            ticker = result.get("ticker")

            if job_id in timed_out_jobs:
                continue

            if job_id in active_jobs:
                process = active_jobs[job_id]["process"]
                process.join(timeout=1)
                del active_jobs[job_id]

            rows = result.get("rows", [])
            all_rows.extend(rows)

            print(f"Completed {ticker}: {len(rows)} MD&A sections extracted")

        now = time.time()

        for job_id, meta in list(active_jobs.items()):
            elapsed = now - meta["start_time"]

            if elapsed > ticker_timeout_seconds:
                ticker = meta["ticker"]
                process = meta["process"]

                process.terminate()
                process.join(timeout=5)

                timed_out_jobs.add(job_id)
                del active_jobs[job_id]

                print(f"Timed out {ticker}")

        time.sleep(0.25)

    mda_df = pd.DataFrame(
        all_rows,
        columns=[
            "ticker",
            "year",
            "filing_url",
            "mda_char_length",
            "mda_text",
        ],
    )

    if not mda_df.empty:
        ticker_order = {
            ticker: order
            for order, ticker in enumerate(tickers)
        }

        mda_df["_ticker_order"] = mda_df["ticker"].map(ticker_order)

        mda_df = mda_df.sort_values(
            by=["_ticker_order", "year"],
            ascending=[True, False],
        )

        mda_df = mda_df.drop(columns=["_ticker_order"])

    mda_df.to_csv(output_filename, index=False)

    print(f"\nDone. Saved file as: {output_filename}")

    return mda_df

if __name__ == "__main__":
    mda_df = run_mda_extraction_clean(
        tickers=tickers,
        n_years=4,
        min_chars=1000,
        max_workers=5,
        ticker_timeout_seconds=300,
        request_timeout_seconds=30,
        absolute_min_mda_chars=250,
        toc_max_chars=1500,
        retrieval_multiplier=5,
        output_filename="mda_full_text.csv",
    )


3) Defining stagnation metrics

In [ ]:
import re
import numpy as np
import pandas as pd
import torch
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sentence_transformers import SentenceTransformer

inno_vocab_df = pd.read_csv("generic_107innovation_detection_phrases.csv")
strategic_vocab_df = pd.read_csv("strategic_phrases.csv")
mda_df = pd.read_csv("mda_full_text.csv")



embed_model = SentenceTransformer("all-MiniLM-L6-v2")
finbert_tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
finbert_model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert")
finbert_model.eval()


#Normalize text for matching
def normalize_match_text(text):
  

    if not isinstance(text, str):
        return ""

    text = text.lower()

    #unify R&D variations
    text = re.sub(r"r\s*&\s*d", "research and development", text)
    text = re.sub(r"r\s*and\s*d", "research and development", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


#Sentence splitting
def split_sentences(text):
    

    if not isinstance(text, str) or not text.strip():
        return []

    text = re.sub(r"\s+", " ", text).strip()

    sentences = re.split(r"(?<=[.!?])\s+", text)

    return [s.strip() for s in sentences if s.strip()]


#Score MD&A text
def score_mda_text(mda_text, phrase_df):
    
    #Validate input text
    if mda_text is None or not isinstance(mda_text, str) or not mda_text.strip():
        return {
            "total_mda_sentences": 0,
            "innovation_sentences": 0,
            "innovation_intensity": 0.0,
            "total_phrase_hits": 0,
            "phrase_hits_per_sentence": 0.0,
        }

    
    #Get unique phrases, drop any missing values, convert to string
    phrases = phrase_df["phrase"].dropna().astype(str).unique()
    if len(phrases) == 0:
        #No phrases to match – return zero scores
        return {
            "total_mda_sentences": 0,
            "innovation_sentences": 0,
            "innovation_intensity": 0.0,
            "total_phrase_hits": 0,
            "phrase_hits_per_sentence": 0.0,
        }

    
    escaped_phrases = [re.escape(phrase) for phrase in phrases]
    pattern_str = "|".join(escaped_phrases)
    phrase_pattern = re.compile(pattern_str, re.IGNORECASE)

    #Scoring logic 
    sentences = split_sentences(mda_text)          
    innovation_sentences = 0
    total_phrase_hits = 0

    for sentence in sentences:
        normalized_sentence = normalize_match_text(sentence)   
        matches = set(m.group(0) for m in phrase_pattern.finditer(normalized_sentence))
        if matches:
            innovation_sentences += 1
            total_phrase_hits += len(matches)

    total_sentences = len(sentences)
    return {
        "total_mda_sentences": total_sentences,
        "innovation_sentences": innovation_sentences,
        "innovation_intensity": (innovation_sentences / total_sentences if total_sentences > 0 else 0.0),
        "total_phrase_hits": total_phrase_hits,
        "phrase_hits_per_sentence": (total_phrase_hits / total_sentences if total_sentences > 0 else 0.0),
    }


#1. Innovation Vocabulary Decay and Strategic Language Decay

def compute_innovation_decay_score(mda_by_year, phrase_df, min_years=3):
   

    if not isinstance(mda_by_year, dict) or not mda_by_year:
        return {
            "yearly_scores": {},
            "slope": None,
            "innovation_decay_score": None,
            "status": "no_data"
        }

    years = sorted(mda_by_year.keys())
    yearly_scores = {}

    #per-year scoring 
    for year in years:
        score = score_mda_text(mda_by_year[year], phrase_df)

        yearly_scores[year] = {
            "total_mda_sentences": score["total_mda_sentences"],
            "innovation_sentences": score["innovation_sentences"],
            "innovation_intensity": score["innovation_intensity"],
            "total_phrase_hits": score["total_phrase_hits"],
            "phrase_hits_per_sentence": score["phrase_hits_per_sentence"],
            "yoy_change": None
        }

    #YoY change 
    for i in range(1, len(years)):
        curr = years[i]
        prev = years[i - 1]

        yearly_scores[curr]["yoy_change"] = (
            yearly_scores[curr]["innovation_intensity"]
            - yearly_scores[prev]["innovation_intensity"]
        )


    valid_years = [
        year for year in years
        if yearly_scores[year]["total_mda_sentences"] > 0
    ]

    if len(valid_years) < min_years:
        return {
            "yearly_scores": yearly_scores,
            "slope": None,
            "innovation_decay_score": None,
            "status": f"insufficient_years_{len(valid_years)}"
        }

    #regression
    x = np.array(valid_years)
    y = np.array([
        yearly_scores[year]["innovation_intensity"]
        for year in valid_years
    ])

    slope = np.polyfit(x, y, 1)[0]

    #decay score = negative slope
    innovation_decay_score = -slope

    return {
        "yearly_scores": yearly_scores,
        "slope": slope,
        "innovation_decay_score": innovation_decay_score,
        "status": "valid"
    }


#2. Topic Rigidity Score

def compute_topic_rigidity(mda_by_year, embed_model):
    #Compute cosine similarity between consecutive years' MD&A texts.

    years = sorted(mda_by_year.keys())

    if len(years) < 2:
        return {
            "similarity_by_year": {},
            "topic_rigidity_score": None
        }

    texts = [mda_by_year[y] for y in years]

    embeddings = embed_model.encode(texts)

    similarity_by_year = {}
    similarities = []

    for i in range(len(years) - 1):
        sim = cosine_similarity(
            [embeddings[i]],
            [embeddings[i + 1]]
        )[0][0]

        next_year = years[i + 1]

        similarity_by_year[next_year] = sim
        similarities.append(sim)

    topic_rigidity_score = np.mean(similarities) if similarities else None

    return {
        "similarity_by_year": similarity_by_year,
        "topic_rigidity_score": topic_rigidity_score
    }

#4. Management Confidence Signal

forward_words = [
    "will", "would", "could", "should",
    "expect", "expects", "expected",
    "anticipate", "anticipates", "anticipated",
    "believe", "believes",
    "intend", "intends", "intended",
    "plan", "plans", "planned",
    "future", "outlook", "guidance",
    "forecast", "forecasts", "forecasted",
    "estimate", "estimates", "estimated"
]


def sentiment_score(text, finbert_tokenizer, finbert_model):
    #Return positive sentiment probability from FinBERT.

    inputs = finbert_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    with torch.no_grad():
        outputs = finbert_model(**inputs)

    probs = torch.nn.functional.softmax(outputs.logits, dim=-1).squeeze().tolist()

    #FinBERT outputs [negative, neutral, positive]
    return probs[2]


def extract_forward_sentences(text, forward_words):
    #Extract sentences containing forward-looking words.

    if not isinstance(text, str) or not text.strip():
        return []

    sentences = re.split(r"(?<=[.!?])\s+", text)

    return [
        sent for sent in sentences
        if any(word in sent.lower() for word in forward_words)
    ]


def compute_management_confidence(mda_by_year, forward_words):
    #Compute average positive sentiment of forward-looking MD&A sentences.

    years = sorted(mda_by_year.keys())
    sentiment_by_year = {}

    for year in years:
        text = mda_by_year[year]
        fwd_sents = extract_forward_sentences(text, forward_words)

        if fwd_sents:
            scores = [sentiment_score(s, finbert_tokenizer, finbert_model) for s in fwd_sents]
            sentiment_by_year[year] = np.mean(scores)
        else:
            sentiment_by_year[year] = 0.5

    management_confidence_score = np.mean(list(sentiment_by_year.values()))

    return {
        "sentiment_by_year": sentiment_by_year,
        "management_confidence_score": management_confidence_score
    }

results = []

ticker_groups = list(mda_df.groupby("ticker", sort=False))
total_tickers = len(ticker_groups)

print(f"Starting NLP metric calculation for {total_tickers} tickers...", flush=True)

for i, (ticker, group) in enumerate(ticker_groups, start=1):
    print(f"[{i}/{total_tickers}] Calculating {ticker}...", flush=True)

    
    mda_by_year = {row.year: row.mda_text for _, row in group.iterrows()}

    #1. Innovation decay
    decay_generic = compute_innovation_decay_score(mda_by_year, inno_vocab_df)

    #2. Strategic decay
    decay_strategic = compute_innovation_decay_score(mda_by_year, strategic_vocab_df)

    #3. Topic rigidity
    rigidity = compute_topic_rigidity(mda_by_year, embed_model)

    #4. Management confidence
    confidence = compute_management_confidence(mda_by_year, forward_words)

    results.append({
        "ticker": ticker,
        "innovation_decay_score": decay_generic["innovation_decay_score"],
        "strategic_decay_score": decay_strategic["innovation_decay_score"],
        "topic_rigidity_score": rigidity["topic_rigidity_score"],
        "management_confidence_score": confidence["management_confidence_score"]
    })

    print(f"[{i}/{total_tickers}] Finished {ticker}", flush=True)

#Final results
nlp_metrics_df = pd.DataFrame(results)
nlp_metrics_df.to_csv("raw_metric_scores.csv", index=False)

print("Done. Saved raw_metric_scores.csv", flush=True)

4) Final stagnation score computation

In [ ]:
INPUT_FILE = "raw_metric_scores.csv"
OUTPUT_FILE = "final_stagnation_scores.csv"


METRIC_COLUMNS = [
    "innovation_decay_score",
    "topic_rigidity_score",
    "strategic_decay_score",
    "management_confidence_score",
]


def zscore(series):
    mean = series.mean(skipna=True)
    std = series.std(skipna=True)

    if std == 0 or pd.isna(std):
        return np.nan

    return (series - mean) / std


def compute_final_scores(df):
    required_cols = ["ticker"] + METRIC_COLUMNS

    missing = [col for col in required_cols if col not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    for col in METRIC_COLUMNS:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df["innovation_decay_zscore"] = zscore(df["innovation_decay_score"])
    df["topic_rigidity_zscore"] = zscore(df["topic_rigidity_score"])
    df["strategic_decay_zscore"] = zscore(df["strategic_decay_score"])
    df["management_confidence_zscore"] = zscore(df["management_confidence_score"])

    #Direction alignment:
    #Higher innovation decay = stronger stagnation signal
    #Higher topic rigidity = stronger stagnation signal
    #Higher strategic decay = stronger stagnation signal
    #Lower management confidence = stronger stagnation signal
    df["management_confidence_zscore_aligned"] = -df["management_confidence_zscore"]

    final_components = [
        "innovation_decay_zscore",
        "topic_rigidity_zscore",
        "strategic_decay_zscore",
        "management_confidence_zscore_aligned",
    ]

    df["final_stagnation_score"] = df[final_components].mean(axis=1, skipna=True)

    df = df.sort_values(
        by="final_stagnation_score",
        ascending=False
    )

    return df


def main():
    print("Loading raw metric scores...")
    df = pd.read_csv(INPUT_FILE)

    print("Standardizing raw metric scores...")
    final_df = compute_final_scores(df)

    final_df.to_csv(OUTPUT_FILE, index=False)

    print(f"Done. Saved final scores to: {OUTPUT_FILE}")


if __name__ == "__main__":
    main()